# HerbVision AI — EfficientDet Training Notebook (Google Colab GPU)

**Project Title**: An Intelligent Deep Learning System for Medicinal Plant Classification and Herbal Remedy Recommendation  
**Authors**: Sankeerthana G, P. Srija, V. Shirisha  
**Guide**: Mr. B. Ravi Raju  
**Department**: Information Technology, Anurag University  

---
### Overview
This notebook trains an **EfficientDet-D0** deep learning model to detect and classify 10 Indian medicinal plant leaf species using transfer learning on pretrained EfficientNet-B0 backbone weights.

## 1. Verify GPU Acceleration
Ensure your Colab runtime is set to **GPU** (`Runtime > Change runtime type > T4 GPU`).

In [ ]:
import tensorflow as tf
print("TensorFlow Version:", tf.__version__)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"SUCCESS: Found {len(gpus)} GPU(s):", gpus)
    !nvidia-smi
else:
    print("WARNING: No GPU detected. Please go to Runtime > Change runtime type > T4 GPU for faster training.")

## 2. Install Required Dependencies

In [ ]:
!pip install -q pillow numpy matplotlib scikit-learn

## 3. Dataset Configuration & Class Mapping
10 Medicinal Plant Classes target: 100–300 leaf images per class.

In [ ]:
CLASSES = [
    "Neem",
    "Tulsi",
    "Aloe Vera",
    "Moringa",
    "Guava",
    "Curry Leaf",
    "Mint",
    "Hibiscus",
    "Eucalyptus",
    "Ashoka"
]

NUM_CLASSES = len(CLASSES)
IMAGE_SIZE = 512
BATCH_SIZE = 16
EPOCHS = 40
LEARNING_RATE = 1e-3
CONFIDENCE_THRESHOLD = 0.50

print(f"Configured {NUM_CLASSES} classes: {CLASSES}")

## 4. Mount Google Drive & Load Dataset
Organize your images into class subdirectories inside Google Drive.

In [ ]:
# Mount Google Drive if dataset is in Drive
# from google.colab import drive
# drive.mount('/content/drive')
# DATASET_DIR = '/content/drive/MyDrive/Medicinal_Leaf_Dataset'

# Sample simulated verification of folder structure
import os
DATASET_DIR = './sample_dataset'
for c in CLASSES:
    os.makedirs(os.path.join(DATASET_DIR, c), exist_ok=True)

print("Ready for dataset ingestion.")

## 5. Build EfficientDet-D0 Architecture (EfficientNet-B0 Backbone + BiFPN)

In [ ]:
from tensorflow.keras import layers, models

def build_bifpn_cell(features, channels=64, name="bifpn"):
    p3_in, p4_in, p5_in, p6_in, p7_in = features
    # Top-down fusion
    p7_td = p7_in
    p6_td = layers.Conv2D(channels, 1, padding='same')(layers.Add()([p6_in, layers.UpSampling2D(2)(p7_td)]))
    p5_td = layers.Conv2D(channels, 1, padding='same')(layers.Add()([p5_in, layers.UpSampling2D(2)(p6_td)]))
    p4_td = layers.Conv2D(channels, 1, padding='same')(layers.Add()([p4_in, layers.UpSampling2D(2)(p5_td)]))
    p3_out = layers.Conv2D(channels, 3, padding='same', activation='swish')(layers.Add()([p3_in, layers.UpSampling2D(2)(p4_td)]))
    # Bottom-up fusion
    p4_out = layers.Conv2D(channels, 3, padding='same', activation='swish')(layers.Add()([p4_in, p4_td, layers.MaxPooling2D(2)(p3_out)]))
    p5_out = layers.Conv2D(channels, 3, padding='same', activation='swish')(layers.Add()([p5_in, p5_td, layers.MaxPooling2D(2)(p4_out)]))
    p6_out = layers.Conv2D(channels, 3, padding='same', activation='swish')(layers.Add()([p6_in, p6_td, layers.MaxPooling2D(2)(p5_out)]))
    p7_out = layers.Conv2D(channels, 3, padding='same', activation='swish')(layers.Add()([p7_in, layers.MaxPooling2D(2)(p6_out)]))
    return [p3_out, p4_out, p5_out, p6_out, p7_out]

def create_efficientdet_model():
    inputs = layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3), name="input_leaf")
    backbone = tf.keras.applications.EfficientNetB0(include_top=False, weights="imagenet", input_tensor=inputs)
    
    p3 = layers.Conv2D(64, 1, padding="same")(backbone.get_layer("block3b_add").output)
    p4 = layers.Conv2D(64, 1, padding="same")(backbone.get_layer("block5c_add").output)
    p5 = layers.Conv2D(64, 1, padding="same")(backbone.get_layer("top_activation").output)
    p6 = layers.Conv2D(64, 3, strides=2, padding="same")(p5)
    p7 = layers.MaxPooling2D(pool_size=3, strides=2, padding="same")(p6)
    
    feats = [p3, p4, p5, p6, p7]
    for i in range(3):
        feats = build_bifpn_cell(feats, channels=64, name=f"bifpn_{i}")
        
    pooled = [layers.GlobalAveragePooling2D()(f) for f in feats]
    fused = layers.Concatenate()(pooled)
    dense = layers.Dense(256, activation="swish")(fused)
    dense = layers.Dropout(0.2)(dense)
    
    class_out = layers.Dense(NUM_CLASSES, activation="softmax", name="class_output")(dense)
    box_out = layers.Dense(4, activation="sigmoid", name="box_output")(dense)
    
    model = models.Model(inputs=inputs, outputs=[class_out, box_out], name="EfficientDet_D0_HerbVision")
    return model

model = create_efficientdet_model()
model.summary()

## 6. Compile & Train Model
Using AdamW optimizer with Cosine Decay and ModelCheckpoint.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=LEARNING_RATE),
    loss={
        "class_output": "sparse_categorical_crossentropy",
        "box_output": tf.keras.losses.Huber()
    },
    loss_weights={"class_output": 1.0, "box_output": 0.5},
    metrics={"class_output": ["accuracy"]}
)
print("Model compiled successfully with classification and regression heads.")

## 7. Export Trained Weights for FastAPI Backend
Exporting as TensorFlow SavedModel. Once exported, download this directory and place it in `backend/models/efficientdet_best/`.

In [ ]:
EXPORT_DIR = "./efficientdet_best"
# tf.saved_model.save(model, EXPORT_DIR)
print(f"Ready to export to {EXPORT_DIR}. Download and copy to backend/models/efficientdet_best.")